In [1]:
import json
import floorplan_backend
import time

In [2]:
import sys
print(sys.version)
print(sys.executable)

3.9.13 (tags/v3.9.13:6de2ca5, May 17 2022, 16:36:42) [MSC v.1929 64 bit (AMD64)]
c:\Users\jairl\Documents\GitHub\FloorplanGenerator\Backend\.venv_rhino\Scripts\python.exe


In [ ]:
with open('./samples/topology.json') as topology_json:
    with open('./samples/construction.json') as construction_json:
        topology = json.load(topology_json)
        construction = json.load(construction_json)
        start = time.time()
        building = floorplan_backend.Building.from_data(topology, construction)
        print(time.time()-start)
        building.solve_joints()
        geoms = [building.layer_geometry(face_id) for face_id in building.face_ids()]

0.00399470329284668


In [ ]:
import numpy as np
import plotly.graph_objects as go

display_material = 'studs'

def polygon_mesh(face):
    face = np.asarray(face)
    n = len(face)
    return (face[:, 0],face[:, 1],face[:, 2],np.zeros(n - 2, dtype=int),np.arange(1, n - 1),np.arange(2, n))

fig = go.Figure()

for geom in geoms:
    colour = np.random.randint(0,255,3)
    colour = f"rgb({colour[0]}, {colour[1]}, {colour[2]})"
    
    for layer in geom:
        for face in [layer['inner'],layer['outer']]:
            x, y, z, i, j, k = polygon_mesh(face)
            
            fig.add_trace(go.Mesh3d(x=x,y=y,z=z,i=i,j=j,k=k,opacity=1,flatshading=True,color=colour,name=layer["name"]))

fig.update_layout(scene=dict(aspectmode="data",xaxis_title="X",yaxis_title="Y",zaxis_title="Z",),)
fig.write_html("geometry.html", auto_open=True)
#fig.show()